# **Model Preparation**

This notebook prepares the engineered feature sets for the experimental phase of the HDFS_v1 anomaly detection study.

The feature engineering notebook created four processed datasets, each representing a different feature representation of HDFS block behavior. Before the statistical baseline and machine learning models can be trained, each feature set must be loaded, cleaned, encoded, and split into consistent training and testing datasets.

The purpose of this notebook is to:

- Load the four engineered feature sets from the `data/processed/` folder.
- Encode the target label into numerical values.
- Separate model input features from the target variable.
- Perform an 80/20 stratified train-test split.
- Create normal-only training data for the Isolation Forest model.
- Save prepared training and testing datasets for later experiments.

The same split process will be applied to all four feature sets so that model results can be compared fairly across feature representations.

In [1]:
# Import required libraries

from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
# Set project paths

current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

processed_dir = project_root / "data" / "processed"
splits_dir = processed_dir / "splits"

splits_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Processed data folder:", processed_dir)
print("Splits folder:", splits_dir)

Project root: c:\Users\taman\Documents\hdfs-anomaly-detection-study
Processed data folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed
Splits folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\splits


## Load Engineered Feature Sets

The four engineered feature sets are loaded from the `data/processed/` folder. Each feature set contains the `BlockId` column for tracking, the `Label` column as the target variable, and the engineered feature columns that will be used during model training.

In [7]:
# Load engineered feature sets

feature_sets = {
    "feature_set_1": pd.read_csv(processed_dir / "feature_set_1.csv"),
    "feature_set_2": pd.read_csv(processed_dir / "feature_set_2.csv"),
    "feature_set_3": pd.read_csv(processed_dir / "feature_set_3.csv"),
    "feature_set_4": pd.read_csv(processed_dir / "feature_set_4.csv"),
}

# Display dimensions for each feature set

for name, df in feature_sets.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns")

feature_set_1: 575,061 rows, 31 columns
feature_set_2: 575,061 rows, 33 columns
feature_set_3: 575,061 rows, 313 columns
feature_set_4: 575,061 rows, 1,084 columns


## Encode Target Labels

The `Label` column contains the ground-truth classification for each HDFS block. Before model training, these labels must be converted into numerical values.

In this study:

- `Success` is encoded as `0`
- `Fail` is encoded as `1`

This encoding allows the models to treat anomaly detection as a binary classification problem.

In [10]:
# Encode target labels for each feature set

label_mapping = {
    "Success": 0,
    "Fail": 1
}

for name, df in feature_sets.items():
    if df["Label"].dtype == "object":
        df["Label"] = df["Label"].map(label_mapping)

In [11]:
# Verify encoded label distribution for each feature set

for name, df in feature_sets.items():
    print(f"\n{name}")
    print(df["Label"].value_counts())


feature_set_1
Label
0    558223
1     16838
Name: count, dtype: int64

feature_set_2
Label
0    558223
1     16838
Name: count, dtype: int64

feature_set_3
Label
0    558223
1     16838
Name: count, dtype: int64

feature_set_4
Label
0    558223
1     16838
Name: count, dtype: int64


## Create Training and Testing Splits

Each engineered feature set is divided into separate training and testing datasets using an 80/20 split.

A stratified split is used to preserve the original class distribution of normal and anomalous HDFS blocks in both datasets. Because the dataset is highly imbalanced, stratification ensures that each model is trained and evaluated on representative samples of both classes.

During this step, the engineered feature columns are separated from the target labels:

- **X** contains the engineered input features.
- **y** contains the encoded target labels.
- **BlockId** is retained separately for tracking predictions back to the original HDFS blocks but is not used during model training.

In [12]:
# Import train/test split

from sklearn.model_selection import train_test_split

In [13]:
# Split each feature set into training and testing datasets

split_data = {}

for name, df in feature_sets.items():

    # Store BlockIds separately for future reference
    block_ids = df["BlockId"]

    # Model input features
    X = df.drop(columns=["BlockId", "Label"])

    # Target labels
    y = df["Label"]

    # Perform an 80/20 stratified split
    (
        X_train,
        X_test,
        y_train,
        y_test,
        block_train,
        block_test,
    ) = train_test_split(
        X,
        y,
        block_ids,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    split_data[name] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "Block_train": block_train,
        "Block_test": block_test,
    }

In [14]:
# Display training and testing dimensions

for name, data in split_data.items():

    print(f"\n{name}")

    print(f"X_train: {data['X_train'].shape}")
    print(f"X_test : {data['X_test'].shape}")

    print(f"y_train: {data['y_train'].shape}")
    print(f"y_test : {data['y_test'].shape}")


feature_set_1
X_train: (460048, 29)
X_test : (115013, 29)
y_train: (460048,)
y_test : (115013,)

feature_set_2
X_train: (460048, 31)
X_test : (115013, 31)
y_train: (460048,)
y_test : (115013,)

feature_set_3
X_train: (460048, 311)
X_test : (115013, 311)
y_train: (460048,)
y_test : (115013,)

feature_set_4
X_train: (460048, 1082)
X_test : (115013, 1082)
y_train: (460048,)
y_test : (115013,)


## Verify Class Distribution

A stratified split was used to preserve the original class distribution in both the training and testing datasets.

Because HDFS anomalies are relatively rare, maintaining approximately the same proportion of normal and anomalous blocks in each split is important for fair model training and evaluation. The following results verify that the class balance was preserved across all four feature sets.

In [15]:
# Display class distribution after the train/test split

for name, data in split_data.items():

    print(f"\n{name}")

    print("\nTraining Set")
    print(data["y_train"].value_counts(normalize=True) * 100)

    print("\nTesting Set")
    print(data["y_test"].value_counts(normalize=True) * 100)


feature_set_1

Training Set
Label
0    97.072045
1     2.927955
Name: proportion, dtype: float64

Testing Set
Label
0    97.071635
1     2.928365
Name: proportion, dtype: float64

feature_set_2

Training Set
Label
0    97.072045
1     2.927955
Name: proportion, dtype: float64

Testing Set
Label
0    97.071635
1     2.928365
Name: proportion, dtype: float64

feature_set_3

Training Set
Label
0    97.072045
1     2.927955
Name: proportion, dtype: float64

Testing Set
Label
0    97.071635
1     2.928365
Name: proportion, dtype: float64

feature_set_4

Training Set
Label
0    97.072045
1     2.927955
Name: proportion, dtype: float64

Testing Set
Label
0    97.071635
1     2.928365
Name: proportion, dtype: float64


## Export Prepared Datasets

The final step in model preparation is to save the training and testing datasets for each engineered feature set.

Each feature set is exported as six separate files:

- **X_train** – Training features used to fit the model.
- **X_test** – Testing features used for evaluation.
- **y_train** – Ground-truth labels for the training data.
- **y_test** – Ground-truth labels for the testing data.
- **Block_train** – Block identifiers corresponding to the training observations.
- **Block_test** – Block identifiers corresponding to the testing observations.

Saving these datasets ensures that every experimental model is trained and evaluated using the exact same observations, allowing fair comparison between the statistical baseline, Logistic Regression, Random Forest, and Isolation Forest.

In [16]:
# Create a folder for each feature set and export the prepared datasets

for name, data in split_data.items():

    # Create the feature set directory
    feature_dir = splits_dir / name
    feature_dir.mkdir(exist_ok=True)

    # Export training features
    data["X_train"].to_csv(
        feature_dir / "X_train.csv",
        index=False
    )

    # Export testing features
    data["X_test"].to_csv(
        feature_dir / "X_test.csv",
        index=False
    )

    # Export training labels
    data["y_train"].to_csv(
        feature_dir / "y_train.csv",
        index=False
    )

    # Export testing labels
    data["y_test"].to_csv(
        feature_dir / "y_test.csv",
        index=False
    )

    # Export training BlockIds
    data["Block_train"].to_csv(
        feature_dir / "Block_train.csv",
        index=False
    )

    # Export testing BlockIds
    data["Block_test"].to_csv(
        feature_dir / "Block_test.csv",
        index=False
    )

In [17]:
# Confirm the exported split files

for name in split_data.keys():

    feature_dir = splits_dir / name

    print(f"\n{name}")

    for file in sorted(feature_dir.glob("*.csv")):
        print(file.name)


feature_set_1
Block_test.csv
Block_train.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv

feature_set_2
Block_test.csv
Block_train.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv

feature_set_3
Block_test.csv
Block_train.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv

feature_set_4
Block_test.csv
Block_train.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv
